## Required Imports

In [8]:
# notebooks/07_batter_data_collection.ipynb

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from pybaseball import statcast_batter, playerid_lookup
from pybaseball import cache
cache.enable()

print("Batter data collection initialized")

Batter data collection initialized


## Define Batter Pool

In [9]:
# Load your existing pitcher data to find frequent opponents
from pitch_analysis import load_clean_data
pitcher_data = load_clean_data()

# Identify batters who appeared most frequently against your pitchers
# The 'batter' column contains the batter MLBAM ID
batter_counts = pitcher_data['batter'].value_counts()

print("=== Most Frequent Batters in Dataset ===")
print(f"Unique batters faced: {len(batter_counts)}")
print(f"\nTop 30 most frequently faced:")
print(batter_counts.head(30))

# Select top 50 batters by frequency for analysis
# These are the batters with most exposure to your pitcher pool
top_batter_ids = batter_counts.head(50).index.tolist()
print(f"\nSelected {len(top_batter_ids)} batters for analysis")

=== Most Frequent Batters in Dataset ===
Unique batters faced: 701

Top 30 most frequently faced:
batter
572233    433
682998    427
668939    409
660271    379
623993    376
606466    374
683002    363
668227    361
650490    358
672695    330
670623    316
646240    309
657041    306
660688    304
553993    304
677951    302
665742    299
543760    298
663624    298
682928    297
663993    296
518692    293
457759    291
666971    291
606192    289
605141    287
656305    285
669257    281
656941    280
608324    279
Name: count, dtype: int64

Selected 50 batters for analysis


## Verify Batter Columns

In [10]:
# Confirm batter column and check what other batter info is available
print("=== Batter-Related Columns in Pitcher Data ===")
batter_cols = [col for col in pitcher_data.columns 
               if 'batter' in col.lower() or 
                  'bat' in col.lower() or
                  'stand' in col.lower()]
print(batter_cols)

# Preview batter data already in pitcher dataset
print("\n=== Sample Batter Records ===")
print(pitcher_data[['batter', 'stand', 'pitch_type', 
                     'events', 'description']].head(10))

=== Batter-Related Columns in Pitcher Data ===
['stand', 'batter', 'bat_score']

=== Sample Batter Records ===
   batter stand pitch_type     events    description
0  518692     L         SI       walk           ball
1  518692     L         CH        NaN           ball
2  518692     L         FC        NaN           ball
3  518692     L         FF        NaN           foul
4  518692     L         CH        NaN           ball
5  605141     R         FC  field_out  hit_into_play
6  605141     R         KC        NaN  called_strike
7  660271     L         KC  field_out  hit_into_play
8  660271     L         CH        NaN           ball
9  660271     L         SI        NaN  called_strike


## Extract Batter Performance from Existing Data

In [11]:
# Extract batter performance against each pitch type from existing data
# This avoids a large additional data pull and uses what you already have

batter_vs_pitch = pitcher_data.groupby(
    ['batter', 'stand', 'pitch_type']
).agg(
    pitches_seen=('pitch_type', 'count'),
    swinging_strikes=('description', lambda x: 
                      (x == 'swinging_strike').sum()),
    called_strikes=('description', lambda x: 
                    (x == 'called_strike').sum()),
    balls_in_play=('events', lambda x: 
                   x.isin(['single', 'double', 'triple', 
                           'home_run', 'field_out', 'grounded_into_double_play',
                           'force_out', 'sac_fly']).sum()),
    hits=('events', lambda x: 
          x.isin(['single', 'double', 'triple', 'home_run']).sum()),
    strikeouts=('events', lambda x: 
                (x == 'strikeout').sum()),
    walks=('events', lambda x: (x == 'walk').sum()),
    home_runs=('events', lambda x: (x == 'home_run').sum())
).reset_index()

# Calculate rates
batter_vs_pitch['whiff_rate'] = (
    batter_vs_pitch['swinging_strikes'] / 
    batter_vs_pitch['pitches_seen']
).round(3)

batter_vs_pitch['contact_rate'] = (
    batter_vs_pitch['balls_in_play'] / 
    batter_vs_pitch['pitches_seen']
).round(3)

batter_vs_pitch['hit_rate'] = (
    batter_vs_pitch['hits'] / 
    batter_vs_pitch['pitches_seen'].clip(lower=1)
).round(3)

# Filter to batters with meaningful sample size
batter_vs_pitch = batter_vs_pitch[
    batter_vs_pitch['pitches_seen'] >= 20
]

print(f"=== Batter vs Pitch Type Performance ===")
print(f"Batter-pitch combinations with 20+ pitches: {len(batter_vs_pitch)}")
print(f"Unique batters: {batter_vs_pitch['batter'].nunique()}")
print(f"\nSample records:")
print(batter_vs_pitch.head(10).to_string(index=False))

=== Batter vs Pitch Type Performance ===
Batter-pitch combinations with 20+ pitches: 1148
Unique batters: 402

Sample records:
 batter stand pitch_type  pitches_seen  swinging_strikes  called_strikes  balls_in_play  hits  strikeouts  walks  home_runs  whiff_rate  contact_rate  hit_rate
 444482     L         CH            20                 2               1              2     1           1      0          0       0.100         0.100     0.050
 444482     L         FF            36                 0               6              9     1           0      0          0       0.000         0.250     0.028
 444482     L         SI            25                 1               7              4     2           1      1          0       0.040         0.160     0.080
 446334     R         FF            37                 2               8              8     3           2      0          1       0.054         0.216     0.081
 453568     L         FF            57                 3              10 

## Pull Batter Names

In [12]:
from pybaseball import playerid_reverse_lookup

# Get names for all batters in your dataset
unique_batter_ids = batter_vs_pitch['batter'].unique().tolist()

print(f"Looking up names for {len(unique_batter_ids)} batters...")

try:
    batter_names = playerid_reverse_lookup(
        unique_batter_ids, 
        key_type='mlbam'
    )
    batter_names = batter_names[['key_mlbam', 'name_first', 'name_last']].copy()
    batter_names['batter_name'] = (batter_names['name_first'] + 
                                    ' ' + batter_names['name_last'])
    batter_names = batter_names.rename(columns={'key_mlbam': 'batter'})
    
    # Merge names into batter performance data
    batter_vs_pitch = batter_vs_pitch.merge(
        batter_names[['batter', 'batter_name']], 
        on='batter', 
        how='left'
    )
    
    print(f"Names resolved for {batter_names.shape[0]} batters")
    print("\nSample with names:")
    print(batter_vs_pitch[['batter_name', 'stand', 'pitch_type', 
                            'pitches_seen', 'whiff_rate', 
                            'hit_rate']].head(10).to_string(index=False))

except Exception as e:
    print(f"Name lookup error: {e}")
    print("Proceeding with batter IDs only")
    batter_vs_pitch['batter_name'] = batter_vs_pitch['batter'].astype(str)

Looking up names for 402 batters...
Names resolved for 402 batters

Sample with names:
     batter_name stand pitch_type  pitches_seen  whiff_rate  hit_rate
   david peralta     L         CH            20       0.100     0.050
   david peralta     L         FF            36       0.000     0.028
   david peralta     L         SI            25       0.040     0.080
   evan longoria     R         FF            37       0.054     0.081
charlie blackmon     L         FF            57       0.053     0.053
charlie blackmon     L         SI            23       0.000     0.087
martín maldonado     R         FF            57       0.175     0.053
martín maldonado     R         SL            32       0.188     0.000
  donovan solano     R         CH            22       0.227     0.045
  donovan solano     R         FC            20       0.100     0.100


## Build Batter Vulnerability Profiles

In [13]:
# For each batter calculate their vulnerability scores by pitch type
# Vulnerability = high whiff rate + low hit rate against a pitch type

# Normalize metrics to 0-1 scale for scoring
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Pivot to get per-batter metrics across pitch types
batter_pivot = batter_vs_pitch.pivot_table(
    index=['batter', 'batter_name', 'stand'],
    columns='pitch_type',
    values=['whiff_rate', 'hit_rate', 'pitches_seen']
).reset_index()

# Flatten column names
batter_pivot.columns = [
    '_'.join(col).strip('_') if col[1] else col[0] 
    for col in batter_pivot.columns
]

print(f"=== Batter Vulnerability Profiles ===")
print(f"Batters profiled: {len(batter_pivot)}")
print(f"Columns available: {list(batter_pivot.columns)}")

=== Batter Vulnerability Profiles ===
Batters profiled: 415
Columns available: ['batter', 'batter_name', 'stand', 'hit_rate_CH', 'hit_rate_CU', 'hit_rate_FC', 'hit_rate_FF', 'hit_rate_FS', 'hit_rate_KC', 'hit_rate_SI', 'hit_rate_SL', 'hit_rate_ST', 'pitches_seen_CH', 'pitches_seen_CU', 'pitches_seen_FC', 'pitches_seen_FF', 'pitches_seen_FS', 'pitches_seen_KC', 'pitches_seen_SI', 'pitches_seen_SL', 'pitches_seen_ST', 'whiff_rate_CH', 'whiff_rate_CU', 'whiff_rate_FC', 'whiff_rate_FF', 'whiff_rate_FS', 'whiff_rate_KC', 'whiff_rate_SI', 'whiff_rate_SL', 'whiff_rate_ST']


## Calculate Pitch Type Vulnerability Scores

In [14]:
# For each pitch type calculate a vulnerability score per batter
# Higher score = batter is more vulnerable to that pitch type

pitch_types = batter_vs_pitch['pitch_type'].unique()
vulnerability_records = []

for _, batter_row in batter_vs_pitch.iterrows():
    # Vulnerability combines whiff rate and inverse hit rate
    # Normalized so both contribute equally
    vulnerability_score = (
        batter_row['whiff_rate'] * 0.6 +      # whiff rate weighted higher
        (1 - batter_row['hit_rate']) * 0.4     # inverse hit rate
    ) * 100

    vulnerability_records.append({
        'batter': batter_row['batter'],
        'batter_name': batter_row['batter_name'],
        'stand': batter_row['stand'],
        'pitch_type': batter_row['pitch_type'],
        'pitches_seen': batter_row['pitches_seen'],
        'whiff_rate': batter_row['whiff_rate'],
        'hit_rate': batter_row['hit_rate'],
        'vulnerability_score': round(vulnerability_score, 2)
    })

vulnerability_df = pd.DataFrame(vulnerability_records)

print("=== Vulnerability Score Distribution ===")
print(vulnerability_df.groupby('pitch_type')['vulnerability_score']
      .describe().round(2))

=== Vulnerability Score Distribution ===
            count   mean   std    min    25%    50%    75%    max
pitch_type                                                       
CH          155.0  45.51  4.87  34.28  41.91  45.22  48.26  59.08
CU           94.0  45.65  5.07  34.28  41.90  45.42  48.68  60.88
FC          122.0  43.71  4.74  30.00  40.18  43.32  46.87  60.88
FF          403.0  44.21  4.50  33.00  40.96  43.74  47.11  59.98
FS            4.0  52.10  3.20  47.50  51.13  53.31  54.28  54.28
KC            1.0  40.00   NaN  40.00  40.00  40.00  40.00  40.00
SI          152.0  39.96  3.42  32.00  38.03  40.00  41.86  55.00
SL          191.0  48.37  5.53  32.40  44.76  48.10  51.91  62.74
ST           26.0  45.85  4.42  39.00  42.90  44.78  49.22  53.62


## Save Batter Data

In [15]:
import os
os.makedirs('../data/processed', exist_ok=True)

batter_vs_pitch.to_parquet(
    '../data/processed/batter_vs_pitch.parquet',
    index=False
)

vulnerability_df.to_parquet(
    '../data/processed/batter_vulnerability.parquet',
    index=False
)

print("=== Batter Data Saved ===")
print(f"batter_vs_pitch.parquet: {len(batter_vs_pitch)} records")
print(f"batter_vulnerability.parquet: {len(vulnerability_df)} records")

=== Batter Data Saved ===
batter_vs_pitch.parquet: 1148 records
batter_vulnerability.parquet: 1148 records
